# NB 00: Setup and Preprocessing

Loads XBRL features, label database, builds the feature matrix, and saves
a preprocessing artifact to Google Drive for downstream notebooks.

**Prerequisites:**
1. Runtime -> Change runtime type -> **T4 GPU** (or A100 if available)
2. Data uploaded to Google Drive at `My Drive/Fin-JEPA/data/` with subdirs `raw/` and `processed/`
3. GitHub PAT stored in Colab Secrets as `GITHUB_PAT`

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import userdata
import os

# Retrieve the GitHub token from Colab Secrets
github_token = userdata.get('GITHUB_PAT')

username = "elroy-galbraith"
repository = "fin-jepa"
repo_url = f"https://{github_token}@github.com/{username}/{repository}.git"

if not os.path.exists(f'/content/{repository}'):
    !git clone {repo_url} /content/{repository}

%cd /content/{repository}

# Install fin-jepa as editable WITHOUT upgrading Colab's pre-installed deps
!pip install -q --no-deps -e '.[dev]'

# Install only the deps Colab doesn't already ship
!pip install -q hydra-core omegaconf mlflow optuna xgboost yfinance rich python-dotenv tqdm

In [ ]:
# Symlink data from Google Drive so relative paths work
import os

DRIVE_DATA = '/content/drive/MyDrive/Fin-JEPA/data'

for subdir in ['raw', 'processed']:
    target = f'data/{subdir}'
    if os.path.islink(target) or os.path.isdir(target):
        !rm -rf {target}
    os.symlink(f'{DRIVE_DATA}/{subdir}', target)

# Verify data exists
!ls -la data/raw/xbrl_features.parquet
!ls -la data/processed/label_database.parquet
!ls -la data/raw/market/market_aligned.parquet
print('Data linked successfully!')

# Verify GPU
import torch
assert torch.cuda.is_available(), 'No GPU detected! Change runtime to T4 or A100.'
print(f'GPU: {torch.cuda.get_device_name(0)}')
props = torch.cuda.get_device_properties(0)
print(f'VRAM: {props.total_memory / 1e9:.1f} GB')

In [ ]:
# Common setup: logging, configs, control flag
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from omegaconf import OmegaConf

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

# Set to True to re-run experiments even if cached results exist
FORCE_RERUN = False

# Load configs from YAML (override individual values below if needed)
benchmark_cfg = OmegaConf.load('configs/study0/benchmark.yaml')
ssl_cfg = OmegaConf.load('configs/study0/ssl_experiment.yaml')

# Paths
RESULTS_DIR = Path('results/study0')
BENCHMARK_PATH = RESULTS_DIR / 'benchmark_results.json'
SWEEP_PATH = RESULTS_DIR / 'ft_sweep_results.json'
SSL_V2_PATH = RESULTS_DIR / 'ssl_experiment_results_v2.json'
FINAL_PATH = RESULTS_DIR / 'final_benchmark.json'

OUTCOMES = ['stock_decline', 'earnings_restate', 'audit_qualification',
            'sec_enforcement', 'bankruptcy']

print(f'Results dir: {RESULTS_DIR}')
print(f'Benchmark cached: {BENCHMARK_PATH.exists()}')
print(f'Sweep cached: {SWEEP_PATH.exists()}')
print(f'SSL v2 cached: {SSL_V2_PATH.exists()}')
print(f'FORCE_RERUN: {FORCE_RERUN}')

# Preprocessing artifact path (shared across split notebooks)
ARTIFACT_DIR = Path('/content/drive/MyDrive/Fin-JEPA/artifacts/study0')
ARTIFACT_PATH = ARTIFACT_DIR / 'preprocessing_v1.pkl'

SEED = int(OmegaConf.select(benchmark_cfg, 'training.seed', default=42))

mask_ratios = [0.15, 0.30, 0.50]
MIN_POSITIVES = 20


In [ ]:
# Shared data loading — used by sweep, SSL, and final benchmark cells
import sys
import os
if '/content/fin-jepa/src' not in sys.path:
    sys.path.append('/content/fin-jepa/src')

from fin_jepa.data.feature_engineering import (
    CATEGORICAL_FEATURES, N_SECTORS, FeatureConfig, build_feature_matrix,
)
from fin_jepa.data.labels import load_label_database
from fin_jepa.data.splits import SplitConfig
from fin_jepa.data.xbrl_loader import load_xbrl_features
from fin_jepa.training.train_study0 import _cfg
from fin_jepa.utils.reproducibility import seed_everything

raw_dir = Path(_cfg(benchmark_cfg, 'data.raw_dir', 'data/raw'))
processed_dir = Path(_cfg(benchmark_cfg, 'data.processed_dir', 'data/processed'))

xbrl_df = load_xbrl_features(raw_dir)
labels_df, _ = load_label_database(processed_dir / 'label_database.parquet')
xbrl_df['period_end'] = pd.to_datetime(xbrl_df['period_end'])
labels_df['period_end'] = pd.to_datetime(labels_df['period_end'])
merged = xbrl_df.merge(labels_df, on=['cik', 'period_end'], how='inner', suffixes=('', '_label'))

split_cfg = SplitConfig(
    train_end=_cfg(benchmark_cfg, 'data.split.train_end', '2017-12-31'),
    val_end=_cfg(benchmark_cfg, 'data.split.val_end', '2019-12-31'),
    test_end=_cfg(benchmark_cfg, 'data.split.test_end', '2023-12-31'),
)
feat_cfg = FeatureConfig(
    use_raw=_cfg(benchmark_cfg, 'features.use_raw', True),
    use_ratios=_cfg(benchmark_cfg, 'features.use_ratios', True),
    use_yoy=_cfg(benchmark_cfg, 'features.use_yoy', True),
    use_sic=_cfg(benchmark_cfg, 'features.use_sic', True),
    use_missingness_flags=_cfg(benchmark_cfg, 'features.use_missingness_flags', True),
    coverage_threshold=_cfg(benchmark_cfg, 'features.coverage_threshold', 0.50),
    normalization_method=_cfg(benchmark_cfg, 'features.normalization_method', 'quantile'),
    median_impute=_cfg(benchmark_cfg, 'features.median_impute', True),
)

universe_df = None
universe_path = raw_dir / 'company_universe.parquet'
if universe_path.exists() and feat_cfg.use_sic:
    universe_df = pd.read_parquet(universe_path)

# ATS-217: seed before build_feature_matrix so QuantileTransformer
# subsampling is deterministic across notebook runs.
SEED = int(_cfg(benchmark_cfg, 'training.seed', 42))
seed_everything(SEED)

splits, scaler, feature_cols, categorical_cols = build_feature_matrix(
    merged, split_cfg, feat_cfg, universe_df=universe_df,
)
n_features = len(feature_cols)
n_cat = len(categorical_cols)
cat_cards = [N_SECTORS] * n_cat if n_cat > 0 else None

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Features: {n_features} continuous + {n_cat} categorical')
print(f'Splits: train={len(splits["train"])}, val={len(splits["val"])}, test={len(splits["test"])}')
print(f'Device: {device}')

# Compute config fingerprint for the feature pipeline
import hashlib
_fp_dict = {
    'split': {'train_end': str(split_cfg.train_end), 'val_end': str(split_cfg.val_end), 'test_end': str(split_cfg.test_end)},
    'features': OmegaConf.to_container(benchmark_cfg.get('features', {}), resolve=True),
}
config_fingerprint = hashlib.sha256(json.dumps(_fp_dict, sort_keys=True).encode()).hexdigest()[:16]
print(f'Config fingerprint: {config_fingerprint}')

In [ ]:
import pickle, hashlib

artifact = {
    'splits': splits,
    'scaler': scaler,
    'feature_cols': feature_cols,
    'categorical_cols': categorical_cols,
    'n_features': n_features,
    'n_cat': n_cat,
    'cat_cards': cat_cards,
    'config_fingerprint': config_fingerprint,
    'created_at': pd.Timestamp.now().isoformat(),
    'benchmark_cfg': OmegaConf.to_container(benchmark_cfg, resolve=True),
    'ssl_cfg': OmegaConf.to_container(ssl_cfg, resolve=True),
}

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
with open(ARTIFACT_PATH, 'wb') as f:
    pickle.dump(artifact, f, protocol=pickle.HIGHEST_PROTOCOL)

size_mb = ARTIFACT_PATH.stat().st_size / 1024 / 1024
print(f'Preprocessing artifact saved to {ARTIFACT_PATH} ({size_mb:.1f} MB)')
print(f'  Config fingerprint: {config_fingerprint}')
print(f'  Train: {len(splits["train"]):,} rows')
print(f'  Val:   {len(splits["val"]):,} rows')
print(f'  Test:  {len(splits["test"]):,} rows')
print(f'  Features: {n_features} ({n_cat} categorical)')